# GIADA52f recovery from600
Restore optimizer and sampling stream, finish601-800 and evaluations. No new warmup, no threshold or architecture change.

In [ ]:
import hashlib,json,os,shutil,subprocess,sys,zipfile,time,threading
from pathlib import Path
REVISION='aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa'
SOURCE_SHA='7e07984c6c8770fee223a83f4694922c2f8cb92529e3df899bc511cf8d5080be'
WORK=Path('/tmp/giada_task52f_resume');WORK.mkdir(exist_ok=False)
repo=WORK/'giada';out=WORK/'result';public=Path('/kaggle/working');log=public/'process.log'
code=None;started=time.time()
def setup(cmd):
 with log.open('a') as stream:subprocess.run(cmd,stdout=stream,stderr=subprocess.STDOUT,check=True)
try:
 assert len(REVISION)==40,'Use console dispatch to pin the exact commit'
 print('[GIADA52f] pinned Torch/GPU setup',flush=True)
 setup([sys.executable,'-m','pip','install','--quiet','--upgrade','torch==2.11.0+cu130','--index-url','https://download.pytorch.org/whl/cu130'])
 for cmd in (['git','clone','--quiet','--depth','1','--branch','codex/surrogate-validity-audit','https://github.com/Zagred47/giada.git',str(repo)],['git','-C',str(repo),'fetch','--quiet','--depth','1','origin',REVISION],['git','-C',str(repo),'checkout','--quiet','--detach','FETCH_HEAD']):setup(cmd)
 sources=list(Path('/kaggle/input').rglob('source41n.zip.bin'));assert len(sources)==1
 with sources[0].open('rb') as f:assert hashlib.file_digest(f,'sha256').hexdigest()==SOURCE_SHA
 checkpoints=list(Path('/kaggle/input').rglob('checkpoint45c.zip.bin'));saved=list(Path('/kaggle/input').rglob('evaluation45d.zip.bin'));assert len(checkpoints)==len(saved)==1
 with checkpoints[0].open('rb') as f:assert hashlib.file_digest(f,'sha256').hexdigest()=='555133b7fc71719ecf459e2b04ffed8fd22a50af198bdb409735e0d8ca996afe'
 with saved[0].open('rb') as f:assert hashlib.file_digest(f,'sha256').hexdigest()=='daabec1f2e998bdc538c2cd7303470d895ad30bedf2124159b1bcad6e2ebc2f7'
 package=list(Path('/kaggle/input').rglob('models52e.zip.bin'));assert len(package)==1
 with package[0].open('rb') as f:assert hashlib.file_digest(f,'sha256').hexdigest()=='8df0d9ec12f5690e20dfee766eedbdf914a2ead984f012280302e69f1692fe49'
 resumes=list(Path('/kaggle/input').rglob('resume52f.zip.bin'));assert len(resumes)==1
 with resumes[0].open('rb') as f:assert hashlib.file_digest(f,'sha256').hexdigest()=='86012728909ac23f20bbce0cabed479d7d6546495b99e41ab095523890d5cf18'
 env=dict(os.environ,CUBLAS_WORKSPACE_CONFIG=':4096:8',MPLBACKEND='Agg')
 with log.open('a') as f:
  proc=subprocess.Popen([sys.executable,'-u',str(repo/'scripts/run_task52f_resume.py'),str(sources[0]),str(checkpoints[0]),str(saved[0]),str(package[0]),str(resumes[0]),str(out)],cwd=repo,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
  timer=threading.Timer(10800,proc.terminate);timer.daemon=True;timer.start()
  for line in proc.stdout:
   f.write(line);f.flush()
   if line.startswith('[GIADA') or 'Error' in line or 'Traceback' in line:print(line.rstrip(),flush=True)
  code=proc.wait();timer.cancel()
finally:
 out.mkdir(exist_ok=True)
 if log.exists():shutil.copy2(log,out/'process.log')
 worker_report=json.loads((out/'final_report.json').read_text()) if (out/'final_report.json').exists() else {}
 (out/'runtime_receipt.json').write_text(json.dumps(dict(revision=REVISION,source_sha256=SOURCE_SHA,worker_exit_code=code,elapsed_seconds=time.time()-started,training_updates_performed=worker_report.get('training_updates'),selection_recomputed=worker_report.get('selection_recomputed',False),thresholds_changed=False)))
 archive=Path(shutil.make_archive(str(public/'task52f_final'),'zip',out))
 with archive.open('rb') as f:digest=hashlib.file_digest(f,'sha256').hexdigest()
 (public/'kaggle_receipt.json').write_text(json.dumps(dict(valid=code==0,execution_valid=code==0,scientific_acceptance_passed=worker_report.get('scientific_acceptance_passed',False),revision=REVISION,source_sha256=SOURCE_SHA,archive=archive.name,sha256=digest,bytes=archive.stat().st_size,worker_exit_code=code,training_updates_performed=worker_report.get('training_updates'),selection_recomputed=worker_report.get('selection_recomputed',False),thresholds_changed=False)))
 print('[GIADA52f] final archive persisted '+digest,flush=True)
assert code==0,'Evaluation recovery failed technically; preserve output and logs'
